## ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YosraHilmi/Fly-Rank-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)



Note on position (data check)

In Google's Search Console bulk export, positions are zero-based (0 = first result), and average position = SUM(sum_position) / SUM(impressions) + 1. The warehouse documentation does not say whether gsc_sum_position follows this convention, so I tested it.

Evidence: the raw ratio SUM(gsc_sum_position) / SUM(gsc_impressions) is below 1 for 2,735 of 176,738 pages (1.55%), with a minimum of 0.0, which is impossible if positions start at 1. Rows with gsc_sum_position = 0 have a median of 1 impression (other rows: 18), which fits a single impression shown in the first result.

Decision: average position = SUM(gsc_sum_position) / SUM(gsc_impressions) + 1, using all rows with impressions. After the fix there are 176,738 rows (one per page) and the minimum average position is 1.0. This is an inference from the data and from Google's documented convention, not a documented fact of this dataset.

# Signal checks

**Signal 1 — Belief: pages at better average positions earn a higher CTR, so a page should be compared with peers at a similar position, not with the whole catalog.
Table: CTR by position band for pages with at least 500 impressions in March 2026 (the cutoff is justified by signal 2). Band = floor of average position; band 20 = position 20 or higher. Median CTR gives each page equal weight; pooled CTR (total clicks / total impressions) weights pages by impressions. n is printed per band. CTR values are percentages.**

In [17]:

from google.colab import userdata
import duckdb

# Retrieve the actual token value
hf_token = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Create the Hugging Face secret using the actual token
con.execute("""
    CREATE SECRET hf_secret (
        TYPE huggingface,
        TOKEN $token
    )
""", {"token": hf_token})

# Query the dataset
rel = "hf://datasets/FlyRank/internship-warehouse"

result = con.execute(f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").fetchone()

print("Total rows:", result[0])



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total rows: 78835655


In [18]:

# Cell 0 - Setup
MONTH = "2026-03"
base = f"{rel}/fact_content_daily_performance"
march_path = f"{base}/month={MONTH}/**/*.parquet"

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret
(TYPE huggingface, TOKEN $token)
""", {"token": hf_token})

# Try the recursive March-only partition path first.
try:
    con.execute(f"""
    CREATE OR REPLACE VIEW fact_mar AS
    SELECT * FROM read_parquet(
        '{march_path}', hive_partitioning=true
    )
    """)
    print("Working fact pattern: March partition recursive glob")
except Exception as e:
    print("First fact pattern failed:", str(e)[:500])
    try:
        march_path = f"{base}/month={MONTH}/*.parquet"
        con.execute(f"""
        CREATE OR REPLACE VIEW fact_mar AS
        SELECT * FROM read_parquet(
            '{march_path}', hive_partitioning=true
        )
        """)
        print("Working fact pattern: March partition direct glob")
    except Exception as e2:
        print("Second fact pattern failed:", str(e2)[:500])
        raise RuntimeError("Could not read the March partition.")

# Try dim_content as a Parquet folder, then as a single Parquet file.
dim_folder = f"{rel}/dim_content/**/*.parquet"
try:
    con.execute(f"""
    CREATE OR REPLACE TABLE dim_content_v AS
    SELECT client_hash_id, content_hash_id, content_type,
           main_intent, is_published, is_deleted
    FROM read_parquet('{dim_folder}')
    """)
    print("Working dim_content pattern: folder Parquet files")
except Exception as e:
    print("First dim_content pattern failed:", str(e)[:500])
    dim_file = f"{rel}/dim_content.parquet"
    try:
        con.execute(f"""
        CREATE OR REPLACE TABLE dim_content_v AS
        SELECT client_hash_id, content_hash_id, content_type,
               main_intent, is_published, is_deleted
        FROM read_parquet('{dim_file}')
        """)
        print("Working dim_content pattern: single Parquet file")
    except Exception as e2:
        print("Second dim_content pattern failed:", str(e2)[:500])
        raise RuntimeError("Could not read dim_content.")

print("setup ok")


Working fact pattern: March partition recursive glob
First dim_content pattern failed: HTTP Error: HTTP GET error on 'https://huggingface.co/api/datasets/FlyRank/internship-warehouse/tree/main/dim_content' (HTTP 404)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Working dim_content pattern: single Parquet file
setup ok


In [19]:

# Compare page-level median CTR with pooled CTR by position band.
display(con.sql("""
WITH eligible AS (
    SELECT
        LEAST(FLOOR(avg_position), 20)::INTEGER AS position_band,
        total_impressions,
        total_clicks,
        ctr
    FROM page_month_v3
    WHERE total_impressions >= 500
      AND avg_position IS NOT NULL
),
band_stats AS (
    SELECT
        position_band,
        COUNT(*) AS n_pages,
        MEDIAN(ctr) AS median_ctr,
        100.0 * SUM(total_clicks)
            / NULLIF(SUM(total_impressions), 0) AS pooled_ctr,
        100.0 * COUNT(*) FILTER (WHERE total_clicks = 0)
            / NULLIF(COUNT(*), 0) AS pct_zero_clicks
    FROM eligible
    GROUP BY position_band
)
SELECT
    position_band,
    n_pages,
    median_ctr,
    pooled_ctr,
    pooled_ctr / NULLIF(median_ctr, 0) AS pooled_to_median_ratio,
    pct_zero_clicks
FROM band_stats
ORDER BY position_band
""").df())

print("CTR values are percentages. The ratio is NULL when median CTR is zero.")
print("Pooled CTR weights pages by impressions; median CTR gives each page equal weight.")


CatalogException: Catalog Error: Table with name page_month_v3 does not exist!
Did you mean "pg_am"?

Observed: the median CTR peaks at positions 3-4 (about 0.27%), falls to about 0.17% across positions 8-19, and to 0.08% from position 20 (n = 11,903). The pooled CTR is almost flat between positions 6 and 19 (0.27%-0.38%), so position explains little in that range. The share of zero-click pages rises with position (8%-10% at positions 3-5, 21%-23% at positions 10-19, 29.7% from position 20). Band 1 is unexpectedly low (median 0.078%, pooled 0.118%, n = 482) even with at least 500 impressions, and three pages hold 21.1% of its impressions; the cause is not explained by this table.

Verdict: **MIXED**. CTR falls as position worsens, but weakly in the middle bands and not at the top. Expected CTR is therefore taken from this data (pooled CTR per band), not from a generic benchmark table.


---



**Signal 2 — Belief: pages with few impressions have an unreliable CTR (mostly zero clicks), so a minimum-volume rule is needed.
Table: impression band, n pages, share of pages with zero clicks, and median CTR (March 2026, all pages). CTR values are percentages.t**

In [ ]:

# Group pages by total impressions.
display(con.sql("""
WITH bucketed AS (
    SELECT
        CASE
            WHEN total_impressions < 50 THEN '<50'
            WHEN total_impressions < 150 THEN '50-149'
            WHEN total_impressions < 500 THEN '150-499'
            WHEN total_impressions < 2000 THEN '500-1999'
            ELSE '2000+'
        END AS impression_band,
        CASE
            WHEN total_impressions < 50 THEN 1
            WHEN total_impressions < 150 THEN 2
            WHEN total_impressions < 500 THEN 3
            WHEN total_impressions < 2000 THEN 4
            ELSE 5
        END AS band_order,
        total_clicks,
        ctr
    FROM page_month_v3
)
SELECT
    impression_band,
    COUNT(*) AS n_pages,
    100.0 * COUNT(*) FILTER (WHERE total_clicks = 0)
        / NULLIF(COUNT(*), 0) AS pct_zero_clicks,
    MEDIAN(ctr) AS median_ctr
FROM bucketed
GROUP BY impression_band, band_order
ORDER BY band_order
""").df())

print("Zero-click percentage is the share of pages with total_clicks = 0.")
print("CTR values are percentages.")


Observed: the share of pages with zero clicks falls steadily as volume rises: 94.9% (<50 impressions, n = 60,624), 83.6% (50-149, n = 24,140), 64.6% (150-499, n = 30,050), 29.3% (500-1,999, n = 32,047) and 4.7% (2,000+, n = 29,877). The median CTR is 0 up to 499 impressions and positive from 500 (0.155%). With a typical CTR near 0.2%, a page needs roughly 500 impressions to expect even one click (rough estimate), so zero clicks at low volume says little about the page itself.

Verdict: **CONFIRMED**. The rule uses a minimum of 500 impressions.


---



## 1. My rule and its reason codes

1. My rule and its reason codes

Plain words: among pages with at least 500 impressions and an average position below 6 in March 2026, compare each page's CTR with the pooled CTR of pages in the same position band. If the page is below it, estimate the unclaimed clicks and rank pages by them. This is a deliberately simple baseline (position only).

- Eligible: total_impressions >= 500 (signal 2: zero-click pages fall from 64.6% at 150-499 impressions to 29.3% at 500-1,999) and average position below 6 (1-based); 21,038 of 176,738 pages (11.9%) pass.
- Expected CTR: pooled CTR of the page's position band = SUM(clicks) / SUM(impressions) x 100.
- Score = total_impressions x max(0, expected_ctr - actual_ctr) / 100 (estimated unclaimed clicks).
- Reason code: CTR_BELOW_POSITION_PEERS. Action label: SNIPPET_REVIEW (review the title and meta description).
- Confidence note: higher at 2,000+ impressions (4.7% zero-click pages), lower at 500-1,999.
- Not used: last_optimized_date and content_updated_date (they describe the export date, July 2026, not March 2026), and any product flags or Analytics columns.

In [ ]:
# *Write the rule in plain words first. Then the reason codes it can output.*
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [ ]:

# Test page-level ratios without adding 1; include zero-position rows.
a = con.sql("""
WITH page_ratios AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_sum_position) / SUM(gsc_impressions) AS raw_position_ratio
    FROM fact_mar
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    MIN(raw_position_ratio) AS min_raw_ratio,
    COUNT(*) FILTER (WHERE raw_position_ratio < 1) AS pages_below_1,
    COUNT(*) AS total_pages,
    100.0 * COUNT(*) FILTER (WHERE raw_position_ratio < 1)
        / NULLIF(COUNT(*), 0) AS pct_pages_below_1
FROM page_ratios
""").df()

b = con.sql("""
SELECT
    MEDIAN(gsc_impressions) FILTER (WHERE gsc_sum_position = 0)
        AS median_impressions_zero_position,
    MEDIAN(gsc_impressions) FILTER (WHERE gsc_sum_position <> 0)
        AS median_impressions_nonzero_position
FROM fact_mar
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

print("Page-level minimum raw position ratio:", a.loc[0, "min_raw_ratio"])
print("Pages with raw ratio below 1:", int(a.loc[0, "pages_below_1"]))
print("Total pages:", int(a.loc[0, "total_pages"]))
print("Percentage below 1:", round(a.loc[0, "pct_pages_below_1"], 2), "%")
print("Median impressions when sum_position = 0:", b.loc[0, "median_impressions_zero_position"])
print("Median impressions when sum_position != 0:", b.loc[0, "median_impressions_nonzero_position"])

if a.loc[0, "min_raw_ratio"] >= 0 and a.loc[0, "pages_below_1"] > 0:
    print("The results are consistent with zero-based positions, but do not prove the warehouse uses that convention.")
else:
    print("The results alone do not establish that the warehouse uses zero-based positions.")


In [ ]:

# Build one row per client + content using all available March GSC rows.
con.execute("""
CREATE OR REPLACE TABLE page_month_v3 AS
SELECT
    f.client_hash_id,
    f.content_hash_id,
    SUM(f.gsc_impressions) AS total_impressions,
    SUM(f.gsc_clicks) AS total_clicks,
    100.0 * SUM(f.gsc_clicks)
        / NULLIF(SUM(f.gsc_impressions), 0) AS ctr,
    SUM(CASE WHEN f.gsc_impressions > 0
             THEN f.gsc_sum_position ELSE 0 END)
        / NULLIF(SUM(CASE WHEN f.gsc_impressions > 0
                          THEN f.gsc_impressions ELSE 0 END), 0)
        + 1 AS avg_position,
    d.content_type,
    d.main_intent
FROM fact_mar f
LEFT JOIN dim_content_v d
    ON f.client_hash_id = d.client_hash_id
   AND f.content_hash_id = d.content_hash_id
WHERE f.gsc_data_available IS TRUE
GROUP BY
    f.client_hash_id, f.content_hash_id,
    d.content_type, d.main_intent
""")

stats = con.sql("""
SELECT
    MIN(avg_position) AS min_avg_position,
    COUNT(*) AS row_count,
    COUNT(DISTINCT content_hash_id) AS distinct_content_count
FROM page_month_v3
""").df()

print("Minimum average position:", stats.loc[0, "min_avg_position"])
print("page_month_v3 row count:", int(stats.loc[0, "row_count"]))
print("Distinct content_hash_id count:", int(stats.loc[0, "distinct_content_count"]))
print("Row count equals distinct content_hash_id count:",
      stats.loc[0, "row_count"] == stats.loc[0, "distinct_content_count"])


In [ ]:

# Include pages with at least 100 impressions; cap position bands at 20.
display(con.sql("""
SELECT
    CAST(LEAST(FLOOR(avg_position), 20) AS INTEGER) AS position_band,
    COUNT(*) AS n_pages,
    MEDIAN(ctr) AS median_ctr,
    QUANTILE_CONT(ctr, 0.25) AS ctr_p25,
    QUANTILE_CONT(ctr, 0.75) AS ctr_p75
FROM page_month_v3
WHERE total_impressions >= 100
  AND avg_position IS NOT NULL
GROUP BY position_band
ORDER BY position_band
""").df())

print("CTR values are percentages; band 20 represents average positions of 20 or higher.")


**Verdict: MIXED.**

Table: median CTR by position band (pages with at least 100 impressions in March 2026; n printed per band).
Observed: from band 3 downward the median CTR falls as position worsens (about 0.25% at positions 3-4, 0.13% at positions 9-10, 0.09% at positions 17-19, and 0 from position 20, n = 25,275). The top bands do not follow this: band 1 has a median of only 0.033% (n = 707) and band 2 0.185% (n = 3,166), below bands 3-5. The reason is not explained by this table. The spread inside a band is wide (band 3: 25th percentile 0.08%, 75th percentile 0.52%), so position explains only part of the CTR. All CTR values are well below published benchmarks (percentages, not fractions), so expected CTR is taken from this data, not from a generic table.


---



In [ ]:

# Eligibility: at least 150 impressions and average position below 6.
display(con.sql("""
SELECT
    COUNT(*) FILTER (
        WHERE total_impressions >= 150 AND avg_position < 6
    ) AS eligible_pages,
    COUNT(*) AS all_pages,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE total_impressions >= 150 AND avg_position < 6
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS eligible_pct
FROM page_month_v3
""").df())

print("Eligibility uses total_impressions >= 150 and avg_position < 6.")


In [ ]:

# Calculate only the combined impression share of the top 3 pages per band.
display(con.sql("""
WITH eligible AS (
    SELECT
        LEAST(FLOOR(avg_position), 20)::INTEGER AS position_band,
        total_impressions
    FROM page_month_v3
    WHERE total_impressions >= 500
      AND avg_position IS NOT NULL
),
ranked AS (
    SELECT
        position_band,
        total_impressions,
        ROW_NUMBER() OVER (
            PARTITION BY position_band
            ORDER BY total_impressions DESC
        ) AS rn,
        SUM(total_impressions) OVER (
            PARTITION BY position_band
        ) AS band_impressions
    FROM eligible
)
SELECT
    position_band,
    ROUND(
        100.0 * SUM(
            CASE WHEN rn <= 3 THEN total_impressions ELSE 0 END
        ) / NULLIF(MAX(band_impressions), 0),
        2
    ) AS top_3_impression_share_pct
FROM ranked
GROUP BY position_band
ORDER BY position_band
""").df())

print("A high top-3 share means a few large pages may strongly influence pooled CTR.")


In [ ]:

# New eligibility rule: at least 500 impressions and average position below 6.
display(con.sql("""
SELECT
    COUNT(*) FILTER (
        WHERE total_impressions >= 500 AND avg_position < 6
    ) AS eligible_pages,
    COUNT(*) AS all_pages,
    ROUND(
        100.0 * COUNT(*) FILTER (
            WHERE total_impressions >= 500 AND avg_position < 6
        ) / NULLIF(COUNT(*), 0),
        2
    ) AS eligible_pct
FROM page_month_v3
""").df())

print("Eligibility rule: total_impressions >= 500 and avg_position < 6.")


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.